# O Neurônio de McCulloch–Pitts

**Capítulo 18** do livro vivo [Ciência de Dados e Aprendizado de Máquina](https://ghdaru.github.io/machinelearning/18-neuronio-artificial.html).

Este notebook roda no Google Colab sem instalar nada — só biblioteca padrão do Python.

Você vai:
1. **Pôr os pesos à mão**, como em 1943;
2. ver o **perceptron encontrá-los sozinho**, como em 1958;
3. tentar o **XOR** e descobrir por que ele parou a área por uma década.


## 1. O neurônio de 1943

Soma ponderada e comparação com um limiar. Só isso.

$$y = 1 \\text{ se } w_1x_1 + w_2x_2 \\geq \\theta, \\quad 0 \\text{ caso contrário}$$

**Repare no que não existe aqui: aprendizado.** Os pesos são postos por você.


In [ ]:
ENTRADAS = [(0, 0), (0, 1), (1, 0), (1, 1)]

FUNCOES = {
    "AND":  lambda a, b: 1 if a and b else 0,
    "OR":   lambda a, b: 1 if a or b else 0,
    "NAND": lambda a, b: 0 if a and b else 1,
    "NOR":  lambda a, b: 0 if a or b else 1,
    "XOR":  lambda a, b: 1 if a != b else 0,   # o impossível
}


class NeuronioMP:
    """McCulloch & Pitts (1943). Não tem `fit` — e isso é deliberado."""

    def __init__(self, pesos, limiar):
        self.pesos, self.limiar = list(pesos), limiar

    def __call__(self, *entradas):
        soma = sum(w * x for w, x in zip(self.pesos, entradas))
        return 1 if soma >= self.limiar else 0


def tabela(neuronio, alvo):
    print(f"  w1={neuronio.pesos[0]:+.2f}  w2={neuronio.pesos[1]:+.2f}  theta={neuronio.limiar:+.2f}")
    print("  x1 x2 | esperado  obtido")
    acertos = 0
    for a, b in ENTRADAS:
        esperado, obtido = alvo(a, b), neuronio(a, b)
        ok = esperado == obtido
        acertos += ok
        print(f"   {a}  {b} |    {esperado}        {obtido}   {'OK' if ok else 'X'}")
    print(f"  -> {acertos}/4")
    return acertos


### Sua vez

Mude `w1`, `w2` e `theta` até a tabela fechar em 4/4. Comece pelo **AND**.

> Dica: pergunte-se quanto a soma vale em cada uma das quatro linhas, e onde
> o corte precisa ficar.


In [ ]:
w1, w2, theta = 0.0, 0.0, 0.0     # <-- MEXA AQUI
FUNCAO = "AND"                     # AND, OR, NAND, NOR, XOR

tabela(NeuronioMP([w1, w2], theta), FUNCOES[FUNCAO])


### Uma solução (não *a* solução)

Existem **infinitas** retas que separam aqueles quatro pontos. As abaixo são umas delas.


In [ ]:
for nome, (pesos, theta) in {
    "AND":  ([1, 1], 2),
    "OR":   ([1, 1], 1),
    "NAND": ([-1, -1], -1),
    "NOR":  ([-1, -1], 0),
}.items():
    print(f"\n{nome}")
    tabela(NeuronioMP(pesos, theta), FUNCOES[nome])


## 2. O perceptron de 1958: a máquina acha os pesos

Rosenblatt acrescentou a peça que faltava — uma **regra de aprendizado**:

$$w \\leftarrow w + \\eta \\,(esperado - obtido)\\, x$$

Mostre um exemplo; se errou, empurre os pesos na direção que teria acertado.
Ele provou que, **se o problema for linearmente separável**, isso converge em
número finito de passos.

A palavra *separável* é o que vai importar daqui a pouco.


In [ ]:
import random


class Perceptron:
    def __init__(self, taxa=0.1, seed=0):
        rng = random.Random(seed)
        self.pesos = [rng.uniform(-0.5, 0.5) for _ in range(2)]
        self.limiar = rng.uniform(-0.5, 0.5)
        self.taxa = taxa
        self.historico = []

    def __call__(self, *entradas):
        soma = sum(w * x for w, x in zip(self.pesos, entradas))
        return 1 if soma >= self.limiar else 0

    def treinar(self, alvo, epocas=100):
        self.historico = []
        for _ in range(epocas):
            erros = 0
            for a, b in ENTRADAS:
                delta = alvo(a, b) - self(a, b)
                if delta:
                    erros += 1
                    self.pesos[0] += self.taxa * delta * a
                    self.pesos[1] += self.taxa * delta * b
                    self.limiar  -= self.taxa * delta
            self.historico.append(erros)
            if erros == 0:
                break
        return self


for nome in ("AND", "OR", "NAND", "NOR"):
    p = Perceptron(seed=1).treinar(FUNCOES[nome])
    print(f"\n{nome}: convergiu em {len(p.historico)} epocas {p.historico}")
    tabela(p, FUNCOES[nome])


## 3. O XOR

Agora tente o XOR. Dê quantas épocas quiser.


In [ ]:
p = Perceptron(seed=1).treinar(FUNCOES["XOR"], epocas=1000)
print(f"convergiu? {p.historico[-1] == 0}")
print(f"erros nas ultimas 10 epocas: {p.historico[-10:]}")
print()
tabela(p, FUNCOES["XOR"])


### O que aconteceu

O número de erros **nem diminui** — o perceptron não se aproxima da solução,
ele oscila. Não há solução de que se aproximar.

Vamos provar por força bruta: varra todos os pesos e limiares numa grade fina
e veja qual é o **melhor resultado possível**.


In [ ]:
melhor, campeao = 0, None
passo = 0.25
faixa = [i * passo for i in range(-12, 13)]

for w1 in faixa:
    for w2 in faixa:
        for theta in faixa:
            n = NeuronioMP([w1, w2], theta)
            acertos = sum(n(a, b) == FUNCOES["XOR"](a, b) for a, b in ENTRADAS)
            if acertos > melhor:
                melhor, campeao = acertos, (w1, w2, theta)

print(f"testadas {len(faixa)**3} combinacoes de (w1, w2, theta)")
print(f"melhor resultado: {melhor}/4  com {campeao}")


**Nunca 4.** E não é falta de resolução da grade: é geometria.

O XOR dispara em (0,1) e (1,0) — **cantos opostos** do quadrado — e não dispara
em (0,0) e (1,1), também opostos. Uma reta divide o plano em dois lados, e
nenhuma reta deixa dois cantos opostos de um lado e os outros dois do outro.

Foi este argumento que Minsky e Papert publicaram em 1969, e que empurrou o
financiamento para a IA simbólica por mais de uma década.

---

## O que vem depois

A saída, popularizada em 1986, **não foi um neurônio melhor**: foi outra camada.
Com uma camada escondida, duas retas se combinam e o XOR se resolve.

É o [capítulo 09 — Redes Multicamadas](https://ghdaru.github.io/machinelearning/09-redes-neurais.html).

E há um laboratório interativo no [capítulo 18](https://ghdaru.github.io/machinelearning/18-neuronio-artificial.html)
onde a reta de decisão se move enquanto você ajusta os pesos.
